In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
GEV fit & diagnostics for block maxima CSVs.

- Reads all CSVs from a directory or glob pattern.
- Expects columns: 'metric' and 'block_max_value' (as emitted by block_maxima.csv).
- Fits SciPy's GEV (genextreme) per metric.
- Saves:
    * <outdir>/gev_<metric>_fit.json       (c, xi=-c, loc, scale, ks_pvalue)
    * <outdir>/gev_<metric>_hist_pdf.png   (histogram + fitted PDF)
    * <outdir>/gev_<metric>_qq.png         (QQ plot)
    * <outdir>/gev_<metric>_return_level.png  (return-level curve + optional bootstrap band)

Usage examples:
    python gev_fit_from_csvs.py --input "/path/to/metrics/*.csv" --outdir /tmp/gev
    python gev_fit_from_csvs.py --input /path/to/metrics_dir --metric application_consumer --n_boot 300
"""

import os, glob, json, argparse, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless environments
import matplotlib.pyplot as plt

from scipy.stats import genextreme, kstest

# ----------------------- helpers -----------------------

def load_block_maxima(input_path: str) -> pd.DataFrame:
    """
    Load all CSV files from a directory or a glob pattern.
    Keep rows that contain both 'metric' and 'block_max_value'.
    """
    paths = []
    if os.path.isdir(input_path):
        # all CSVs in directory (non-recursive by default); make recursive if you like
        paths = sorted([os.path.join(input_path, p) for p in os.listdir(input_path) if p.endswith(".csv")])
    else:
        # treat as glob
        paths = sorted(glob.glob(input_path))

    if not paths:
        raise FileNotFoundError(f"No CSV files found for: {input_path}")

    frames = []
    for p in paths:
        try:
            df = pd.read_csv(p)
        except Exception as e:
            print(f"[WARN] Skipping unreadable CSV {p}: {e}")
            continue
        if {"metric", "block_max_value"}.issubset(df.columns):
            frames.append(df[["metric", "block_max_value"]].copy())
        else:
            # silently ignore unrelated CSVs (e.g., merge_metrics.csv)
            pass

    if not frames:
        raise ValueError("No CSVs contained the required columns: 'metric', 'block_max_value'.")

    out = pd.concat(frames, ignore_index=True)
    # Clean data
    out["block_max_value"] = pd.to_numeric(out["block_max_value"], errors="coerce")
    out = out.dropna(subset=["block_max_value"])
    return out


def fit_gev(values: np.ndarray):
    """
    Fit SciPy's GEV (genextreme) to data and compute KS p-value.
    Returns a dict with c, xi=-c, loc, scale, ks_pvalue, n.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 5:
        raise ValueError("Not enough data to fit GEV (need >= 5).")

    # MLE fit: (c, loc, scale). Classical xi = -c
    c, loc, scale = genextreme.fit(values)
    D, p = kstest(values, 'genextreme', args=(c, loc, scale))

    return {
        "n": int(values.size),
        "shape_c": float(c),
        "xi": float(-c),
        "location_mu": float(loc),
        "scale_beta": float(scale),
        "ks_D": float(D),
        "ks_pvalue": float(p),
    }


def return_level(c: float, loc: float, scale: float, T: np.ndarray) -> np.ndarray:
    """
    Return level z_T for return period T (years, blocks, etc.),
    where non-exceedance p = 1 - 1/T.
    SciPy uses 'c'; classical xi = -c. Handles the Gumbel limit when c ~ 0.
    """
    T = np.asarray(T, dtype=float)
    p = 1.0 - 1.0 / T
    # Quantile using SciPy's parameterization:
    # x_p = loc + (scale/c) * ((-ln p)^(-c) - 1), with Gumbel limit at c->0: x_p = loc - scale*ln(-ln p)
    eps = 1e-7
    if abs(c) < 1e-6:
        # Gumbel
        q = loc - scale * np.log(-np.log(np.clip(p, eps, 1 - eps)))
    else:
        q = loc + (scale / c) * ((-np.log(np.clip(p, eps, 1 - eps))) ** (-c) - 1.0)
    return q


def bootstrap_ci(values: np.ndarray, n_boot=200, seed=0, Ts=(2, 5, 10, 20, 50, 100)):
    """
    Lightweight bootstrap for uncertainty bands on return levels.
    Returns dict with percentiles for return levels and for parameters.
    """
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    n = values.size
    Ts = np.asarray(Ts, dtype=float)

    c_list, loc_list, scale_list = [], [], []
    rl_boot = []

    for _ in range(n_boot):
        sample = rng.choice(values, size=n, replace=True)
        try:
            c_b, loc_b, scale_b = genextreme.fit(sample)
        except Exception:
            continue
        c_list.append(c_b); loc_list.append(loc_b); scale_list.append(scale_b)
        rl_boot.append(return_level(c_b, loc_b, scale_b, Ts))

    if not c_list:
        return None

    params = {
        "shape_c": np.array(c_list),
        "xi": -np.array(c_list),
        "location_mu": np.array(loc_list),
        "scale_beta": np.array(scale_list),
    }
    rl_boot = np.vstack(rl_boot)  # (n_eff_boot, len(Ts))

    def pct(a, lo=2.5, hi=97.5):
        return np.nanpercentile(a, [lo, 50.0, hi], axis=0)

    rl_ci = pct(rl_boot)         # 3 x len(Ts)
    params_ci = {k: pct(v) for k, v in params.items()}

    return {"Ts": Ts, "return_levels_ci": rl_ci, "params_ci": params_ci}


# ----------------------- plotting -----------------------

def plot_hist_with_pdf(values, c, loc, scale, out_png):
    xs = np.linspace(np.nanmin(values), np.nanpercentile(values, 99.5), 400)
    pdf = genextreme.pdf(xs, c, loc=loc, scale=scale)

    plt.figure(figsize=(8, 5))
    plt.hist(values, bins="auto", density=True, alpha=0.6, edgecolor="black")
    plt.plot(xs, pdf, linewidth=2)
    plt.title("Histogram with fitted GEV PDF")
    plt.xlabel("Block maxima")
    plt.ylabel("Density")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()


def plot_qq(values, c, loc, scale, out_png):
    n = len(values)
    vs = np.sort(values)
    probs = (np.arange(1, n + 1) - 0.5) / n
    theo = genextreme.ppf(probs, c, loc=loc, scale=scale)

    plt.figure(figsize=(5, 5))
    plt.scatter(theo, vs, s=12)
    mn = min(vs.min(), theo.min()); mx = max(vs.max(), theo.max())
    plt.plot([mn, mx], [mn, mx])
    plt.title("GEV QQ plot")
    plt.xlabel("Theoretical quantiles")
    plt.ylabel("Empirical quantiles")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()


def plot_return_level(c, loc, scale, out_png, boot=None):
    Ts = np.logspace(np.log10(1.5), np.log10(200), 200)  # smooth curve
    rl = return_level(c, loc, scale, Ts)

    plt.figure(figsize=(8, 5))
    plt.plot(Ts, rl, linewidth=2, label="Fitted RL")

    if boot is not None and boot.get("return_levels_ci") is not None:
        Ts_boot = boot["Ts"]
        low, med, hi = boot["return_levels_ci"]
        plt.fill_between(Ts_boot, low, hi, alpha=0.25, label="Bootstrap 95% CI")
        plt.plot(Ts_boot, med, linestyle="--", linewidth=1.5, label="Bootstrap median")

    plt.xscale("log")
    plt.xlabel("Return period T (blocks)")
    plt.ylabel("Return level")
    plt.title("Return level plot (GEV)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()


# ----------------------- main -----------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True,
                    help="Directory containing CSVs or a glob pattern (e.g., /metrics/*.csv)")
    ap.add_argument("--outdir", required=True, help="Where to write plots and JSON")
    ap.add_argument("--metric", choices=["consumer_producer", "application_consumer"],
                    help="If set, fit only this metric; else fit all present in CSVs.")
    ap.add_argument("--min_blocks", type=int, default=10,
                    help="Minimum # of block maxima required to fit (default: 10)")
    ap.add_argument("--n_boot", type=int, default=0,
                    help="Bootstrap iterations for uncertainty bands (0 to disable; try 200 for light CI)")
    ap.add_argument("--seed", type=int, default=0, help="Random seed for bootstrap")
    args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        df = load_block_maxima(args.input)
        # either filter by requested metric or do all
        metrics = sorted(df["metric"].dropna().unique().tolist())
        if args.metric:
            metrics = [m for m in metrics if m == args.metric]
            if not metrics:
                raise ValueError(f"Requested metric '{args.metric}' not found in input.")

        for metric in metrics:
            series = df.loc[df["metric"] == metric, "block_max_value"].dropna().astype(float).values
            series = series[np.isfinite(series)]
            if series.size < args.min_blocks:
                print(f"[WARN] Skipping metric '{metric}' (only {series.size} blocks; need >= {args.min_blocks})")
                continue

            # Fit
            fit = fit_gev(series)
            c, loc, scale = fit["shape_c"], fit["location_mu"], fit["scale_beta"]
            print(f"[INFO] {metric}: n={fit['n']}  c={c:.6f}  xi={fit['xi']:.6f}  "
                  f"mu={loc:.6f}  beta={scale:.6f}  KS p={fit['ks_pvalue']:.4f}")

            # Optional bootstrap
            boot = None
            if args.n_boot and args.n_boot > 0:
                try:
                    boot = bootstrap_ci(series, n_boot=args.n_boot, seed=args.seed)
                except Exception as e:
                    print(f"[WARN] Bootstrap failed for {metric}: {e}")

            # Save JSON
            out_json = os.path.join(args.outdir, f"gev_{metric}_fit.json")
            payload = {"metric": metric, **fit}
            if boot is not None:
                payload["bootstrap"] = {
                    "params_ci_percentiles": {
                        # arrays: [2.5%, 50%, 97.5%]
                        "shape_c": boot["params_ci"]["shape_c"].tolist(),
                        "xi": boot["params_ci"]["xi"].tolist(),
                        "location_mu": boot["params_ci"]["location_mu"].tolist(),
                        "scale_beta": boot["params_ci"]["scale_beta"].tolist(),
                    },
                    "return_levels_Ts": boot["Ts"].tolist(),
                    "return_levels_ci_percentiles": boot["return_levels_ci"].tolist(),
                }
            with open(out_json, "w") as f:
                json.dump(payload, f, indent=2)

            # Plots
            plot_hist_with_pdf(series, c, loc, scale,
                               os.path.join(args.outdir, f"gev_{metric}_hist_pdf.png"))
            plot_qq(series, c, loc, scale,
                    os.path.join(args.outdir, f"gev_{metric}_qq.png"))
            plot_return_level(c, loc, scale,
                              os.path.join(args.outdir, f"gev_{metric}_return_level.png"),
                              boot=boot)

if __name__ == "__main__":
    main()


In [ ]:
python gev_fit_from_csvs.py --input 